In [ ]:
%pip install plotly umap-learn sqlalchemy psycopg2-binary

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sqlalchemy import create_engine
from umap import UMAP

# --- Connection ---
# If you're running this notebook inside the docker-compose Jupyter container,
# the DB host should be the compose service name: pgvector
# If you're running locally, use localhost.
DB_HOST = os.getenv("PGHOST", "pgvector")
DB_PORT = os.getenv("PGPORT", "5432")
DB_USER = os.getenv("PGUSER", "postgres")
DB_PASSWORD = os.getenv("PGPASSWORD", "postgres")
DB_NAME = os.getenv("PGDATABASE", "vector_db")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# --- Load rows from ingestion table ---
query = """
SELECT
  id,
  book_id,
  chunk_id,
  page_start,
  page_end,
  position,
  created_at,
  content,
  embedding
FROM documents
"""

df = pd.read_sql(query, engine)
print(f"Loaded {len(df)} rows from documents")


def _parse_embedding(value):
    """Return a 1D numpy array for the embedding, or None if invalid."""
    if value is None:
        return None

    # pgvector drivers often return list/tuple already
    if isinstance(value, (list, tuple, np.ndarray)):
        arr = np.asarray(value, dtype=float)
        return arr

    # Sometimes it can arrive as a string like "[0.1, 0.2, ...]"
    if isinstance(value, str):
        s = value.strip()
        try:
            parsed = ast.literal_eval(s)
            arr = np.asarray(parsed, dtype=float)
            return arr
        except Exception:
            # fallback: strip brackets and split
            s2 = s.strip("[]")
            if not s2:
                return None
            try:
                arr = np.asarray([float(x) for x in s2.split(",")], dtype=float)
                return arr
            except Exception:
                return None

    return None


emb_list = df["embedding"].apply(_parse_embedding)
valid_mask = emb_list.apply(lambda x: x is not None and x.ndim == 1)
invalid = (~valid_mask).sum()
if invalid:
    print(f"Dropping {invalid} rows with invalid embeddings")

df = df.loc[valid_mask].copy()
emb_list = emb_list.loc[valid_mask]

# Ensure consistent dimensions
dims = emb_list.apply(lambda x: int(x.shape[0]))
mode_dim = int(dims.mode().iloc[0]) if len(dims) else 0
consistent_mask = dims == mode_dim
inconsistent = (~consistent_mask).sum()
if inconsistent:
    print(f"Dropping {inconsistent} rows with inconsistent dims")

df = df.loc[consistent_mask].copy()
emb_list = emb_list.loc[consistent_mask]

X = np.vstack(emb_list.values) if len(df) else np.zeros((0, 0), dtype=float)
print(f"Embedding matrix shape: {X.shape}")

# Add short preview for hover
if "content" in df.columns:
    df["content_preview"] = df["content"].fillna("").str.replace("\n", " ").str.slice(0, 200)
else:
    df["content_preview"] = ""

In [ ]:
# --- Reduce to 3D (UMAP) ---
# If you have very few points, UMAP can fail or be unstable.
# In that case we fallback to simple strategies.

if X.shape[0] == 0:
    raise RuntimeError("No valid embeddings loaded from documents.")

if X.shape[0] < 5:
    print(f"Only {X.shape[0]} points. Skipping UMAP; using a simple fallback.")
    if X.shape[1] >= 3:
        coords = X[:, :3]
    elif X.shape[1] == 2:
        coords = np.column_stack([X, np.zeros((X.shape[0], 1))])
    else:
        coords = np.column_stack([X, np.zeros((X.shape[0], 2))])
else:
    reducer = UMAP(n_components=3, random_state=42)
    coords = reducer.fit_transform(X)

plot_df = df.copy()
plot_df["x"] = coords[:, 0]
plot_df["y"] = coords[:, 1]
plot_df["z"] = coords[:, 2]

print(plot_df[["book_id", "chunk_id", "page_start", "page_end", "created_at", "x", "y", "z"]].head())

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"  # or "notebook" / "notebook_connected"
print("Current renderer:", pio.renderers.default)

In [ ]:
# --- Build an interactive 3D plot (one trace per book) ---
books = sorted(plot_df["book_id"].dropna().unique().tolist())
print(f"Books found: {books}")

fig = go.Figure()

for b in books:
    sub = plot_df[plot_df["book_id"] == b].sort_values(["chunk_id", "position"], na_position="last")

    hover = (
        "book_id=%{customdata[0]}<br>"
        "chunk_id=%{customdata[1]}<br>"
        "pages=%{customdata[2]}-%{customdata[3]}<br>"
        "created_at=%{customdata[4]}<br>"
        "preview=%{customdata[5]}"
        "<extra></extra>"
    )

    customdata = np.column_stack(
        [
            sub["book_id"].astype(str).values,
            sub["chunk_id"].astype(str).values,
            sub["page_start"].fillna("").astype(str).values,
            sub["page_end"].fillna("").astype(str).values,
            sub["created_at"].astype(str).values,
            sub["content_preview"].astype(str).values,
        ]
    )

    fig.add_trace(
        go.Scatter3d(
            x=sub["x"],
            y=sub["y"],
            z=sub["z"],
            mode="markers",
            name=b,
            customdata=customdata,
            hovertemplate=hover,
            marker=dict(size=3),
            visible=True,
        )
    )

# Dropdown: show all or only one book
buttons = []

# All
buttons.append(
    dict(
        label="All",
        method="update",
        args=[{"visible": [True] * len(books)}],
    )
)

# One book
for idx, b in enumerate(books):
    vis = [False] * len(books)
    vis[idx] = True
    buttons.append(
        dict(
            label=b,
            method="update",
            args=[{"visible": vis}],
        )
    )

fig.update_layout(
    title="Book chunks in 3D embedding space (UMAP)",
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3",
    ),
    legend_title_text="book_id (click to toggle)",
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.0,
            y=1.1,
        )
    ],
    margin=dict(l=0, r=0, t=60, b=0),
)
print(f"About to show figure for: {books}")
fig.show()

In [ ]:
# Tip: multi-select books
# - Use the dropdown to focus on a single book.
# - To compare multiple books, keep the dropdown on "All" and click entries in the legend
#   to hide/show specific books (this is Plotly's built-in multi-toggle).
